# EXP-018 Colab 실행용 (이거만 쓰면 됨)

## 시작 전 30초
1. 메뉴 **런타임 → 런타임 유형 변경 → GPU → 저장**
2. 아래 셀을 **위에서부터 하나씩** 실행 (Shift+Enter)
3. 빨간 에러가 나면 그 셀에서 멈추고 메시지 확인

## 주의
- smoke 숫자는 성적 아님
- `notebooks/best` 에 저장하지 않음 (Drive `MyDrive/exp018/` 만 사용)
- test split 사용 안 함

기준선 EXP-015 Exact@5 = **53.24%**

## ① 한 번에 준비 (Drive + 코드 + 패키지 + 데이터)

실행하면 Drive 연결 → 깃 클론 → 설치 → 데이터 확인까지 합니다.
비공개 레포면 GitHub 토큰을 한 번 물어봅니다.

In [ ]:
import os
import subprocess
from getpass import getpass
from pathlib import Path

from google.colab import drive

BRANCH = "cursor/exp018-phase-a-prep-41a8"
REPO = Path("/content/sooyah")
DRIVE = Path("/content/drive/MyDrive/exp018")
SMOKE_OUT = DRIVE / "_smoke"
OUT_A = DRIVE / "018a"
OUT_B = DRIVE / "018b"
OUT_C = DRIVE / "018c"
EXP015_PRED = DRIVE / "exp015_sillok_ft_valid_predictions.csv"


def sh(cmd: str, check: bool = True):
    print("\n>>>", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"명령 실패 (exit={r.returncode}): {cmd}")
    return r.returncode


# 1) Drive
drive.mount("/content/drive")
for p in (SMOKE_OUT, OUT_A, OUT_B, OUT_C):
    p.mkdir(parents=True, exist_ok=True)
print("Drive OK:", DRIVE)

# 2) Clone
if REPO.exists():
    sh(f"rm -rf {REPO}", check=False)

url_public = f"https://github.com/google-ajou-Epigraph/sooyah.git"
code = sh(f"git clone -b {BRANCH} {url_public} {REPO}", check=False)
if code != 0:
    print("공개 클론 실패 → GitHub 토큰 필요 (Settings → Developer settings → PAT, repo 권한)")
    token = getpass("GitHub token (ghp_...): ").strip()
    if not token:
        raise RuntimeError("토큰이 비어 있습니다.")
    url_auth = f"https://{token}@github.com/google-ajou-Epigraph/sooyah.git"
    sh(f"git clone -b {BRANCH} {url_auth} {REPO}")

os.chdir(REPO)
print("cwd:", os.getcwd())

# 3) deps + LFS data
sh("python -m pip -q install -r requirements.txt")
sh('git lfs pull --include="data/model_inputs/artificial_mask_dataset.csv"')

# 4) CUDA
sh("python work/check_cuda_env.py")

mask = Path("data/model_inputs/artificial_mask_dataset.csv")
assert mask.is_file() and mask.stat().st_size > 1_000_000, "데이터 LFS pull 실패"
print("데이터 OK:", mask, "bytes=", mask.stat().st_size)
print("EXP-015 CSV:", "있음" if EXP015_PRED.is_file() else "없음 → 다음 셀에서 업로드")
print("\n① 준비 완료")

## ② EXP-015 예측 CSV (McNemar용)

- 이미 Drive `MyDrive/exp018/exp015_sillok_ft_valid_predictions.csv` 가 있으면 **이 셀 건너뛰어도 됨**
- 없으면 실행 → **파일 선택** 창에서 PC의 `handoff/exp018/exp015_sillok_ft_valid_predictions.csv` 고르기

In [ ]:
from pathlib import Path
from google.colab import files

EXP015_PRED = Path("/content/drive/MyDrive/exp018/exp015_sillok_ft_valid_predictions.csv")
EXP015_PRED.parent.mkdir(parents=True, exist_ok=True)

if EXP015_PRED.is_file() and EXP015_PRED.stat().st_size > 1000:
    print("이미 있음:", EXP015_PRED, "bytes=", EXP015_PRED.stat().st_size)
else:
    print("CSV 파일을 선택하세요 (exp015_sillok_ft_valid_predictions.csv)")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("업로드된 파일이 없습니다.")
    name = next(iter(uploaded))
    EXP015_PRED.write_bytes(uploaded[name])
    print("저장 완료:", EXP015_PRED, "bytes=", EXP015_PRED.stat().st_size)

## ③ GPU smoke (짧은 테스트, 수치 무시)

파이프라인만 확인. **성공하면 ④로.**

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir("/content/sooyah")
SMOKE_OUT = Path("/content/drive/MyDrive/exp018/_smoke")

cmd = (
    "python -m src.data.finetune_sillokbert "
    "--run-mode smoke "
    "--planned-experiment-id EXP-018 "
    f"--output-dir {SMOKE_OUT} "
    "--epochs 1 --batch-size 8 --learning-rate 5e-5"
)
print(cmd)
r = subprocess.run(cmd, shell=True)
assert r.returncode == 0, "smoke 실패"
print("smoke OK (수치는 공식 아님)")

## ④ 정식 학습 EXP-018A (lr=5e-5, 4 epoch)

**오래 걸림.** 끝날 때까지 기다리세요. 끊기면 맨 아래 Resume 셀 사용.

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir("/content/sooyah")
OUT = Path("/content/drive/MyDrive/exp018/018a")
LOG = Path("/content/drive/MyDrive/exp018/018a_train.log")
OUT.mkdir(parents=True, exist_ok=True)

cmd = (
    "python -m src.data.finetune_sillokbert "
    "--run-mode full --planned-experiment-id EXP-018A "
    f"--output-dir {OUT} "
    "--epochs 4 --batch-size 16 --learning-rate 5e-5"
)
print(cmd)
with open(LOG, "w", encoding="utf-8") as logf:
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="")
        logf.write(line)
    rc = p.wait()
assert rc == 0, f"018A 실패 exit={rc}"
print("018A 완료 →", OUT)

## ⑤ 정식 학습 EXP-018B (lr=3e-5, 4 epoch)

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir("/content/sooyah")
OUT = Path("/content/drive/MyDrive/exp018/018b")
LOG = Path("/content/drive/MyDrive/exp018/018b_train.log")
OUT.mkdir(parents=True, exist_ok=True)

cmd = (
    "python -m src.data.finetune_sillokbert "
    "--run-mode full --planned-experiment-id EXP-018B "
    f"--output-dir {OUT} "
    "--epochs 4 --batch-size 16 --learning-rate 3e-5"
)
print(cmd)
with open(LOG, "w", encoding="utf-8") as logf:
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="")
        logf.write(line)
    rc = p.wait()
assert rc == 0, f"018B 실패 exit={rc}"
print("018B 완료 →", OUT)

## ⑥ 정식 학습 EXP-018C (lr=2e-5, 4 epoch)

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir("/content/sooyah")
OUT = Path("/content/drive/MyDrive/exp018/018c")
LOG = Path("/content/drive/MyDrive/exp018/018c_train.log")
OUT.mkdir(parents=True, exist_ok=True)

cmd = (
    "python -m src.data.finetune_sillokbert "
    "--run-mode full --planned-experiment-id EXP-018C "
    f"--output-dir {OUT} "
    "--epochs 4 --batch-size 16 --learning-rate 2e-5"
)
print(cmd)
with open(LOG, "w", encoding="utf-8") as logf:
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="")
        logf.write(line)
    rc = p.wait()
assert rc == 0, f"018C 실패 exit={rc}"
print("018C 완료 →", OUT)

## ⑦ Exact@5 선정 + subsample 2000 평가 (A/B/C 자동)

학습 3개가 끝난 뒤 이 셀 한 번 실행.

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir("/content/sooyah")
EXP015 = Path("/content/drive/MyDrive/exp018/exp015_sillok_ft_valid_predictions.csv")

runs = [
    ("EXP-018A", Path("/content/drive/MyDrive/exp018/018a")),
    ("EXP-018B", Path("/content/drive/MyDrive/exp018/018b")),
    ("EXP-018C", Path("/content/drive/MyDrive/exp018/018c")),
]

for exp_id, run_dir in runs:
    print("\n====", exp_id, run_dir, "====")
    assert run_dir.exists(), f"학습 폴더 없음: {run_dir}"

    cmd1 = (
        "python -m src.eval.select_checkpoint_by_exact "
        f"--run-dir {run_dir} --run-mode smoke --max-samples 2000 --seed 42 "
        f"--batch-size 32 --experiment-id {exp_id} --split valid"
    )
    print(cmd1)
    assert subprocess.run(cmd1, shell=True).returncode == 0

    out = run_dir / "eval_chain"
    cmd2 = (
        "python -m src.eval.run_ft_eval_chain "
        f"--run-dir {run_dir} --experiment-id {exp_id} --skip-selection "
        f"--model-path {run_dir / 'best_exact5'} --batch-size 32 "
        f"--exp015-predictions {EXP015} --output-dir {out}"
    )
    print(cmd2)
    assert subprocess.run(cmd2, shell=True).returncode == 0
    print("저장:", out)

print("\n⑦ 완료. gate 통과 run만 아래 full valid 셀 실행.")

## ⑧ (조건부) full valid — 게이트 통과한 run만

`eval_chain` 결과에서 allow_full_valid / Exact@5가 +0.7%p & p&lt;0.05 인 것만.
아래에서 `RUN` 과 `EXP_ID` 만 고치고 실행.

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir("/content/sooyah")

# 예: 통과한 것만 바꾸기
EXP_ID = "EXP-018A"
RUN = Path("/content/drive/MyDrive/exp018/018a")
EXP015 = Path("/content/drive/MyDrive/exp018/exp015_sillok_ft_valid_predictions.csv")
OUT = RUN / "eval_chain_full"

cmd = (
    "python -m src.eval.run_ft_eval_chain "
    f"--run-dir {RUN} --experiment-id {EXP_ID} --skip-selection "
    f"--model-path {RUN / 'best_exact5'} --batch-size 32 --run-full-valid "
    f"--exp015-predictions {EXP015} --output-dir {OUT}"
)
print(cmd)
assert subprocess.run(cmd, shell=True).returncode == 0
print("full valid 완료 →", OUT)

## (선택) 세션 끊김 Resume

`EPOCH_DONE` = 이미 끝난 epoch 번호. 예: epoch 2까지 있으면 2.

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir("/content/sooyah")

EXP_ID = "EXP-018A"
LR = "5e-5"
OUT = Path("/content/drive/MyDrive/exp018/018a")
EPOCH_DONE = 2  # 여기만 수정

cmd = (
    "python -m src.data.finetune_sillokbert "
    f"--run-mode full --planned-experiment-id {EXP_ID} "
    f"--output-dir {OUT} --epochs 4 --batch-size 16 --learning-rate {LR} "
    f"--resume-from {OUT}/checkpoint-epoch-{EPOCH_DONE}"
)
print(cmd)
# 필요할 때만 주석 해제
# assert subprocess.run(cmd, shell=True).returncode == 0

## 끝 체크

Drive `MyDrive/exp018/` 에 각 `018a`/`018b`/`018c` 마다:
- `checkpoint-epoch-1` … `4` 전부
- `best/` , `best_exact5/`
- `run_meta.json` , `finetune_metrics.json` , `exact5_selection_summary.json`
- `eval_chain/`

**`notebooks/best` 로 복사하지 말 것.**